# DE EIA - Réseaux de neurones

2025-26 - Bordeaux INP - ENSEIRB-MATMECA

Michaël Clément

## TP2 - Réseaux de neurones multicouches (MLP)

Dans ce TP, nous allons explorer les réseaux de neurones multicouches (Multi-Layer Perceptron, MLP) et comprendre leur intérêt par rapport au perceptron simple.
Nous commencerons par résoudre un problème classique de classification non linéaire (XOR), puis nous observerons les risques de surapprentissage sur un jeu de données simple.
Enfin, nous appliquerons un MLP à un cas plus réaliste : la classification d'images de chiffres avec MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Classifieur non linéaire pour la fonction XOR

Dans cette première partie, nous allons étudier un cas simple mais fondamental : la fonction XOR.  
Ce problème ne peut pas être résolu par un classifieur linéaire comme le perceptron.  
Nous allons voir qu'un MLP avec une couche cachée permet de résoudre ce type de problème grâce à la non-linéarité introduite par les fonctions d'activation.

### Dataset

In [ ]:
# Création du dataset
n_samples = 500
X = np.random.randint(low=0, high=2, size=(n_samples, 2)).astype('float32') 
y = np.logical_xor(X[:, 0], X[:, 1]).astype('int')
X += 0.1 * np.random.randn(n_samples, 2) # Ajout de bruit pour rendre le problème plus difficile

# Visualisation du dataset
def plot_dataset(X, y):
    plt.figure(figsize=(6, 6))
    plt.scatter(X[:, 0][y == 0], X[:, 1][y == 0], label="Class 0 (False)")
    plt.scatter(X[:, 0][y == 1], X[:, 1][y == 1], label="Class 1 (True)")
    plt.xlabel(r"$x_1$"); plt.ylabel(r"$x_2$")
    plt.legend()
plot_dataset(X, y)

# Conversion en tenseurs PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

### Modèle MLP

Compléter la classe ci-dessous pour mettre en place un réseau de neurones MLP avec une couche cachée (2 neurones) et une couche de sortie (1 neurone).


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

model = MLP()

In [ ]:
# Autre version possible : création directe avec `nn.Sequential`
# YOUR CODE HERE
raise NotImplementedError()

### Entraînement du modèle

In [ ]:
model = MLP()

n_epochs = 2000
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
# Choix de la loss function
# YOUR CODE HERE
raise NotImplementedError()

def train(model, X, y, n_epochs, loss_fn, optimizer):
    loss_history = []
    for epoch in range(n_epochs):
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        loss_history.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f"[epoch {epoch:03d}] loss = {loss.item():.4f}")
    return loss_history

loss_history = train(model, X_tensor, y_tensor, n_epochs, loss_fn, optimizer)

### Frontière de décision

In [ ]:
def plot_decision_boundary(model, X, y):
    # Définition d'une grille de points
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    # Prédiction du modèle pour l'ensemble de la grille et seuillage
    with torch.no_grad():
        y_pred = model(grid).reshape(xx.shape)
    # Affichage de la frontière et du dataset
    plt.figure(figsize=(6, 6))
    plt.contourf(xx, yy, y_pred, alpha=0.3, levels=1, colors=['C0', 'C1'])
    plt.scatter(X[:, 0][y == 0], X[:, 1][y == 0], label='Classe 0')
    plt.scatter(X[:, 0][y == 1], X[:, 1][y == 1], label='Classe 1')
    plt.xlabel("x1"); plt.ylabel("x2")
    plt.show()

plot_decision_boundary(model, X, y)

## 2. Capacité du modèle et surapprentissage (*overfitting*)

Bien que le MLP soit un outil puissant, capable d'apprendre des frontières de décision non linéaires, nous allons voir que si le modèle est trop complexe par rapport à la tâche, il peut apprendre parfaitement les données d’entraînement, mais échouer à généraliser sur des données nouvelles.

Nous allons mettre en place une séparation entre un ensemble d'entraînement et un ensemble de test, et nous analyserons l'évolution des performances sur chacun.
Cela permettra de visualiser concrètement le surapprentissage (*overfitting*) et de souligner l'importance de l’évaluation.

### Dataset

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# Fonction pour générer le dataset avec ensembles de train et test
def generate_noisy_circles():
    # Dataset de deux cercles concentriques, correspondant à deux classes
    X, y = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=SEED)
    # Séparation en ensembles de train et test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEED)
    # Ajout de bruit de labels sur les exemples de train (pour illustrer l'overfitting)
    n_flip = int(0.15 * len(y_train))
    flip_indices = np.random.choice(len(y_train), size=n_flip, replace=False)
    y_train[flip_indices] = 1 - y_train[flip_indices]
    return X_train, y_train, X_test, y_test

# Fonction pour afficher la frontière de décision sur les ensembles de train et de test
def plot_decision_boundary(model, X_train, y_train, X_test, y_test):
    x_min, x_max = X_train[:, 0].min() - 0.1, X_train[:, 0].max() + 0.1
    y_min, y_max = X_train[:, 1].min() - 0.1, X_train[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        y_preds = model(grid).reshape(xx.shape)
        y_preds = (y_preds >= 0.5)
    # Visualisation de ensembles de train et test
    plt.figure(figsize=(8, 4))
    plt.subplot(121)
    plt.contourf(xx, yy, y_preds > 0.5, alpha=0.3, levels=1, colors=['C0', 'C1'])
    plt.scatter(X_train[:, 0][y_train == 0], X_train[:, 1][y_train == 0])
    plt.scatter(X_train[:, 0][y_train == 1], X_train[:, 1][y_train == 1])
    plt.title('Ensemble de train')
    plt.subplot(122)
    plt.contourf(xx, yy, y_preds > 0.5, alpha=0.3, levels=1, colors=['C0', 'C1'])
    plt.scatter(X_test[:, 0][y_test == 0], X_test[:, 1][y_test == 0])
    plt.scatter(X_test[:, 0][y_test == 1], X_test[:, 1][y_test == 1])
    plt.title('Ensemble de test')
    plt.tight_layout()

In [ ]:
# Génération des ensembles de train et test
X_train, y_train, X_test, y_test = generate_noisy_circles()

# Visualisation de ensembles de train et test
plt.figure(figsize=(8, 4))
plt.subplot(121)
plt.scatter(X_train[:, 0][y_train == 0], X_train[:, 1][y_train == 0])
plt.scatter(X_train[:, 0][y_train == 1], X_train[:, 1][y_train == 1])
plt.title('Ensemble de train')
plt.subplot(122)
plt.scatter(X_test[:, 0][y_test == 0], X_test[:, 1][y_test == 0])
plt.scatter(X_test[:, 0][y_test == 1], X_test[:, 1][y_test == 1])
plt.title('Ensemble de test')
plt.tight_layout()

### Entraînement de plusieurs modèles

In [ ]:
# Fonction pour entraîner un modèle
def train(model, X_train, y_train, X_test, y_test, n_epochs, loss_fn, optimizer):
    # Conversion en tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
    for epoch in range(n_epochs+1):
        # Optimisation sur le train
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = loss_fn(y_pred, y_train)
        loss.backward()
        optimizer.step()
        # Evaluation sur le train et le test
        if epoch % (n_epochs//10) == 0:
            with torch.no_grad():
                train_acc = (y_pred.round() == y_train).float().mean().item()
                test_acc = (model(X_test).round() == y_test).float().mean().item()
                print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Train acc: {train_acc:.4f} | Test acc: {test_acc:.4f}")

In [ ]:
# Modèle à forte capacité qui risque de surapprendre le bruit
model_overfitting = nn.Sequential(
    nn.Linear(2, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1),
    nn.Sigmoid()
)

# Entraînement du modèle
n_epochs = 2000
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_overfitting.parameters(), lr=0.01)
train(model_overfitting, X_train, y_train, X_test, y_test, n_epochs, loss_fn, optimizer)

# Affichage de la frontière de décision
plot_decision_boundary(model_overfitting, X_train, y_train, X_test, y_test)

In [ ]:
# Modèle plus simple qui devrait généraliser
model_simple = nn.Sequential(
    nn.Linear(2, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.Sigmoid()
)

# Entraînement du modèle
n_epochs = 1000
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_simple.parameters(), lr=0.01)
train(model_simple, X_train, y_train, X_test, y_test, n_epochs, loss_fn, optimizer)

# Affichage de la frontière de décision
plot_decision_boundary(model_simple, X_train, y_train, X_test, y_test)

## 3. Application pratique sur le dataset MNIST

Dans cette dernière partie, nous allons appliquer ce que nous avons vu précédemment à un problème plus réaliste : la classification d'images de chiffres manuscrits à l'aide du dataset MNIST.
L'objectif est de construire un réseau de neurones simple (MLP) capable de reconnaître les chiffres de 0 à 9 à partir d'images de taille 28x28.
Nous verrons au passage comment :
- charger un dataset standard avec PyTorch,
- utiliser un modèle à plusieurs couches pour traiter des images,
- évaluer les performances sur un jeu de test.

### Dataset

In [ ]:
import torchvision

# Chargement du dataset MNIST
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True)

# Récupération des images et des labels
X_train, y_train = train_dataset.data, train_dataset.targets
X_test, y_test = train_dataset.data, train_dataset.targets
print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')

# Normalisation des pixels dans l'intervalle [0, 1]
X_train = X_train / 255.
X_test = X_test / 255.

In [ ]:
# Affichage de quelques exemples
plt.figure(figsize=(6, 6))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i].item()}")
    plt.axis('off')
plt.tight_layout()

Afin d'utiliser les images 28x28 de MNIST dans notre contexte, il faut les "aplatir" au format vecteur compatible avec le réseau de neurones MLP. Pour cela, nous allons simplement placer tous les pixels les uns à la suite des autres dans un vecteur de taille 784 (28x28).

Par ailleurs, le dataset MNIST étant beaucoup plus grand que les données synthétiques utilisées auparavant, il est nécessaire d'utiliser l'entraînenement par batch.
Pour cela, il faut encapsuler les données dans un `DataLoader` en PyTorch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Aplatissement des images 2D au format vecteur
# YOUR CODE HERE
raise NotImplementedError()

# Encapsulation des données dans un `TensorDataset`, puis dans un `DataLoader`
batch_size = 64
train_dataloader = DataLoader(TensorDataset(X_train_flatten, y_train), batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(TensorDataset(X_test_flatten, y_test), batch_size=batch_size, shuffle=False)

# Parcours de quelques batchs
i = 0
for (X_batch, y_batch) in train_dataloader:
    print(f'{X_batch.shape=}')
    print(f'{y_batch.shape=}')
    i = i + 1
    if i >= 3:
        break

### Création d'un modèle et entraînement

Dans cette étape, vous allez créer un modèle MLP simple adapté à la classification des images MNIST, qui contient 10 classes.
Le modèle ne devra pas être trop profond : deux à trois couches cachées devraient suffir.

Ensuite, vous mettrez en place une boucle d’entraînement utilisant une fonction de perte adaptée à la classification multi-classes, ainsi qu’un optimiseur pour ajuster les poids du réseau.

Vous pouvez essayer différentes architectures de réseau ainsi que différentes valeurs d'hyperparamètres.

Pensez à monitorer l'évolution de la loss et des performances au fur et à mesure de l'entrainement, en vous inspirant du code utilisé pour les parties précédentes.

In [ ]:
# Mettre en place un réseau de neurones MLP pour la classification des images MNIST
class MLP_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
# Création du modèle
# YOUR CODE HERE
raise NotImplementedError()

# Définition de la loss function et de l'optimizer
# YOUR CODE HERE
raise NotImplementedError()

n_epochs = 10

for e in range(n_epochs):
    
    # Apprentissage : parcours de l'ensemble de train (par batch)
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    for (X_batch, y_batch) in train_dataloader:
        # Passage des données dans le modèle et optimisation des paramètres
        # YOUR CODE HERE
        raise NotImplementedError()
        # Monitoring de la loss et de l'accuracy
        running_train_loss += loss.item() * X_batch.size(0)
        _, predicted_train = torch.max(y_pred.data, 1)
        total_train += y_batch.size(0)
        correct_train += (predicted_train == y_batch).sum().item()
    epoch_train_loss = running_train_loss / len(train_dataloader.dataset)
    epoch_train_accuracy = 100 * correct_train / total_train
        
    # Evaluation : parcours de l'ensemble de test (par batch)
    model.eval()
    total_test = 0
    correct_test = 0
    with torch.no_grad():
        for (X_batch, y_batch) in test_dataloader:
            # Passage des données dans le modèle et évaluation de l'accuracy
            y_pred = model(X_batch)
            _, predicted_test = torch.max(y_pred.data, 1)
            total_test += y_batch.size(0)
            correct_test += (predicted_test == y_batch).sum().item()
    test_accuracy = 100 * correct_test / total_test

    print(f"epoch [{e+1}/{n_epochs}], train_loss = {epoch_train_loss:.4f}, train_accuracy = {epoch_train_accuracy:.2f}%, test_accuracy = {test_accuracy:.2f}%")

Essayez d'interpréter et de commenter les résultats. Votre modèle est-il en surapprentissage ?

Quelle est la taille de votre modèle, c'est-à-dire, combien a-t-il de paramètres ?

Trouvez des exemples d'images mal classées par le modèle. Qu'en pensez-vous ?